# Building MultipoleExpansion with vmap

The real power of BFEs in JAX is being able to `vmap` over anything.
This notebook shows how to build `MultipoleExpansion` objects inside vmapped functions,
sweeping over parameters like axis ratio `q` or density profile parameters,
while keeping coordinates fixed or vmapped separately.

In [1]:
import time
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from bfeax import MultipoleExpansion

In [2]:
def nfw_density(x, y, z, rs=10.0, q=0.5):
    """Oblate NFW with axis ratio q."""
    r_tilde = jnp.sqrt(x**2 + y**2 + (z / q)**2)
    s = r_tilde / rs
    return 1.0 / (s * (1.0 + s)**2)

grid_kw = dict(r_min=1e-2, r_max=300.0, n_r=64, l_max=8)

## 1. Build inside a traced function

Define a function that takes a parameter (like `q`) and builds a `MultipoleExpansion`
from a parametrized density. When we trace this through `jax.jit` with a fixed `q`,
the entire build pipeline is compiled and cached.

In [3]:
@jax.jit
def build_and_force(xyz, q):
    """Build expansion for axis ratio q, evaluate force at xyz."""
    x, y, z = xyz
    dens_func = lambda x, y, z: nfw_density(x, y, z, rs=10.0, q=q)
    exp = MultipoleExpansion.from_density(dens_func, **grid_kw, prune_modes=False,symmetry='axisymmetric')
    return exp.force(x, y, z)

# Test: single call with fixed q
xyz = jnp.array([1.0, 2.0, 3.0])
q = 0.5

print("Single call (q=0.5):")
force = build_and_force(xyz, q)
print(f"  Force: {force}")

%timeit jax.block_until_ready(build_and_force(xyz, q))

Single call (q=0.5):
  Force: (Array(-5.34103433, dtype=float64), Array(-10.68206866, dtype=float64), Array(-27.68351472, dtype=float64))
218 μs ± 95.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 2. vmap over q (axis ratio)

Build many `MultipoleExpansion` objects for different axis ratios in a single vmapped call.
The coordinates are held constant (`in_axes=(None, 0)`).

In [9]:
xyz = jnp.array([1.0, 2.0, 3.0])
q_values = jnp.linspace(0.5, 1., 50)

print(f"Evaluating force at {xyz} for q = {q_values}")
print()


# created vmapped function
mapped_force = jax.jit(jax.vmap(build_and_force, in_axes=(None, 0)))
_ = mapped_force(xyz, jnp.ones_like(q_values))  # warmup call to compile the vmapped function
# vmap over q, keep xyz constant
# in_axes=(None, 0) means: xyz is constant (axis=None), q varies along axis 0

t0 = time.perf_counter()
all_forces =  jax.block_until_ready(mapped_force(xyz, q_values)) # shape: (n_q, 3, n_points)
t1 = time.perf_counter()
# print the time in ms
print(f"Time for vmapped force evaluation over 50 q-values: {(t1 - t0) * 1000:.2f} ms")

Evaluating force at [1. 2. 3.] for q = [0.5        0.51020408 0.52040816 0.53061224 0.54081633 0.55102041
 0.56122449 0.57142857 0.58163265 0.59183673 0.60204082 0.6122449
 0.62244898 0.63265306 0.64285714 0.65306122 0.66326531 0.67346939
 0.68367347 0.69387755 0.70408163 0.71428571 0.7244898  0.73469388
 0.74489796 0.75510204 0.76530612 0.7755102  0.78571429 0.79591837
 0.80612245 0.81632653 0.82653061 0.83673469 0.84693878 0.85714286
 0.86734694 0.87755102 0.8877551  0.89795918 0.90816327 0.91836735
 0.92857143 0.93877551 0.94897959 0.95918367 0.96938776 0.97959184
 0.98979592 1.        ]

Time for vmapped force evaluation over 50 q-values: 25.39 ms


In [11]:
# Benchmark against agama
import agama
def oblate_NFW_density(xyz, rs=10.0, q=0.5):
    """Oblate NFW with axis ratio q."""
    x, y, z = xyz[:,0], xyz[:,1], xyz[:,2]
    r_tilde = np.sqrt(x**2 + y**2 + (z / q)**2)
    s = r_tilde / rs
    return 1.0 / (s * (1.0 + s)**2)

def agama_nfw_force(x, y, z, rs=10.0, q=0.5):
    """Oblate NFW with axis ratio q."""
    # Compute the potential and force using agama
    potential = agama.Potential(type="Multipole", density=oblate_NFW_density,symmetry='axisymmetric', gridSizeR=64, rmin=1e-2, rmax=300, lmax=8)
    force = potential.force(x, y, z)
    return force

xyz_a = np.array([1.0, 2.0, 3.0])
# Benchmark agama
t0 = time.perf_counter()
force_agama = agama_nfw_force(xyz_a[0], xyz_a[1], xyz_a[2], rs=10.0, q=0.5)
t1 = time.perf_counter()
print(f"Time for single force evaluation with agama: {(t1 - t0) * 1000:.2f} ms")

# now loop over the 20 q-values and evaluate the force with agama
print(f"Evaluating force at {xyz_a} for q = {q_values} with agama")
t0 = time.perf_counter()
forces_agama = []
for q in q_values:
    force_agama = agama_nfw_force(xyz_a[0], xyz_a[1], xyz_a[2], rs=10.0, q=q)
    forces_agama.append(force_agama)
t1 = time.perf_counter()
print(f"Time for evaluating force over 50 q-values with agama: {(t1 - t0) * 1000:.2f} ms")


Time for single force evaluation with agama: 7.84 ms
Evaluating force at [1. 2. 3.] for q = [0.5        0.51020408 0.52040816 0.53061224 0.54081633 0.55102041
 0.56122449 0.57142857 0.58163265 0.59183673 0.60204082 0.6122449
 0.62244898 0.63265306 0.64285714 0.65306122 0.66326531 0.67346939
 0.68367347 0.69387755 0.70408163 0.71428571 0.7244898  0.73469388
 0.74489796 0.75510204 0.76530612 0.7755102  0.78571429 0.79591837
 0.80612245 0.81632653 0.82653061 0.83673469 0.84693878 0.85714286
 0.86734694 0.87755102 0.8877551  0.89795918 0.90816327 0.91836735
 0.92857143 0.93877551 0.94897959 0.95918367 0.96938776 0.97959184
 0.98979592 1.        ] with agama
Time for evaluating force over 50 q-values with agama: 105.10 ms


/var/folders/mn/slwdcx9x3lv6wky78mz09pkw0000gp/T/ipykernel_18027/3272964657.py:8: RuntimeWarning: divide by zero encountered in divide
  return 1.0 / (s * (1.0 + s)**2)
